#### WebAssembly vs RISC-V

The characteristics of WebAssembly are:
1. Stack architecture
2. Byte code
3. Statically typed
4. Block-structured
5. Non-uniform store

For each of these:
- explain briefly,
- give one or more WebAssembly instructions to illustrate your explanation.

1. **Stack architecture.** WebAssembly has no user-visible registers; every operand lives on an implicit operand stack. Instructions pop their inputs from the top of the stack and push their result. `i32.add` pops the top two `i32` values and pushes their sum — there is no destination register to name. Compare with RISC-V's `add rd, rs1, rs2`, which explicitly names three registers.

2. **Byte code.** The compiled form is a compact binary encoding: each opcode is a single byte plus variable-length (LEB128) immediates. The `.wat` textual form is translated to this binary by `wat2wasm`. For example, `i32.const 5` assembles to the two bytes `0x41 05` (opcode `0x41` for `i32.const`, then the LEB128 encoding of `5`). RISC-V uses fixed 32-bit instruction words instead.

3. **Statically typed.** Every stack operand carries a type from `{i32, i64, f32, f64}`, and every local, global, function parameter, and function result has a declared type. The module is validated before execution: the validator rejects code that would combine mismatched types (e.g. `f32.add` on `i32` operands) or leave the stack in an inconsistent state at a join. Typing is visible in instruction mnemonics (`i32.add` vs `f64.add`) and in declarations such as `(local $x i32)` and `(func $p (param i32) (result i32) …)`.

4. **Block-structured.** Control flow is expressed by nested constructs — `block L … end`, `loop L … end`, and `if … else … end` — with branches targeting labels by nesting depth rather than arbitrary instruction addresses. `br L` and `br_if L` can only jump to an enclosing label. For example, a `while E do S` compiles to `(loop $L code(E); if code(S) br $L end end)`, where `br $L` re-enters the loop head; the validator guarantees every branch lands inside a well-formed block.

5. **Non-uniform store.** The machine has four separate address spaces — code, linear memory, globals, and the operand stack / frame locals — each with its own access instructions. Locals use `local.get $x` / `local.set $x`; module globals use `global.get $y` / `global.set $y`; byte-addressable memory uses `i32.load offset=n` / `i32.store offset=n`; immediates enter the operand stack via `i32.const n`. RISC-V, by contrast, has a single uniform memory and a single register file.

---

As a reminder, here are some WebAssembly instructions:

| instruction           | effect                                             | trap condition   |
|:----------------------|:---------------------------------------------------|:-----------------|
| `i32.const i`         | `s[sp], sp := i, sp + 1`                           | `sp < StackSize` |
| `i32.add`             | `s[sp - 2], sp := s[sp - 2] + s[sp - 1], sp - 1`   |                  |
| `i32.sub`             | `s[sp - 2], sp := s[sp - 2] - s[sp - 1], sp - 1`   |                  |
| `i32.mul`             | `s[sp - 2], sp := s[sp - 2] × s[sp - 1], sp - 1`   |                  |
| `i32.div_s`           | `s[sp - 2], sp := s[sp - 2] div s[sp - 1], sp - 1` | `s[sp - 1] = 0`  |
| `i32.rem_s`           | `s[sp - 2], sp := s[sp - 2] mod s[sp - 1], sp - 1` | `s[sp - 1] = 0`  |
| `i32.eqz`             | `s[sp - 1] := s[sp - 1] = 0`                       |                  |
| `i32.eq`              | `s[sp - 2], sp := s[sp - 2] = s[sp - 1], sp - 1`   |                  |
| `i32.ne`              | `s[sp - 2], sp := s[sp - 2] ≠ s[sp - 1], sp - 1`   |                  |
| `i32.lt_s`            | `s[sp - 2], sp := s[sp - 2] < s[sp - 1], sp - 1`   |                  |
| `i32.gt_s`            | `s[sp - 2], sp := s[sp - 2] > s[sp - 1], sp - 1`   |                  |
| `i32.le_s`            | `s[sp - 2], sp := s[sp - 2] ≤ s[sp - 1], sp - 1`   |                  |
| `i32.ge_s`            | `s[sp - 2], sp := s[sp - 2] ≥ s[sp - 1], sp - 1`   |                  |
| `i32.load offset = n` | `s[sp - 1] := m[s[sp - 1] + n]`                    | `0 ≤ s[sp - 1] + n < MemSize` |
| `i32.store offset = n`| `m[s[sp - 2] + n], sp := s[sp - 1], sp - 2`        | `0 ≤ s[sp - 1] + n < MemSize` |
| `local.get x`         | `s[sp], sp := s[loc(x)], sp + 1`                   |                  |
| `local.set x`         | `s[loc(x)], sp := s[sp - 1], sp - 1`               |                  |
| `global.get x`        | `s[sp], sp := g[loc(x)], sp + 1`                   |                  |
| `global.set x`        | `g[loc(x)], sp := s[sp - 1], sp - 1`               |                  |